In [19]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

In [49]:
df_pm = pd.read_parquet('/Users/eric/Desktop/cali_air_quality/data/pm25_hourly_with_smoke_fire.parquet')
df_met = pd.read_parquet('/Users/eric/Desktop/cali_air_quality/data/meteorological_hourly_raw')

In [50]:
df_pm['datetime'] = pd.to_datetime(df_pm['datetime'])
df_met['datetime'] = pd.to_datetime(
    df_met['date_local'].astype(str) + ' ' + df_met['time_local'].astype(str)
)

In [51]:
df_wind = df_met.pivot_table(
    index=['site_id', 'datetime'],
    columns='parameter',
    values='sample_measurement'
).reset_index()

In [52]:
df_wind.columns.name = None
df_wind = df_wind.rename(columns={
    'Wind Speed - Resultant': 'wind_speed',
    'Wind Direction - Resultant': 'wind_dir'
})

### spatial mapping

In [53]:
met_locs = df_met.groupby('site_id')[['latitude', 'longitude']].first().reset_index()
pm_locs = df_pm.groupby('site_id')[['latitude', 'longitude']].first().reset_index()

In [54]:
met_coords = np.radians(met_locs[['latitude', 'longitude']].values)
pm_coords = np.radians(pm_locs[['latitude', 'longitude']].values)

finding nearest neighbor

In [55]:
tree = BallTree(met_coords, metric='haversine')
dist_rad, ind = tree.query(pm_coords, k=1)

earth_radius_km = 6371
dist_km = dist_rad * earth_radius_km

site_map = {}
dist_map = {}

for i, pm_site in enumerate(pm_locs['site_id']):
    nearest_met_site = met_locs.iloc[ind[i][0]]['site_id']
    distance = dist_km[i][0]
    
    site_map[pm_site] = nearest_met_site
    dist_map[pm_site] = distance

In [56]:
df_pm['met_site_id'] = df_pm['site_id'].map(site_map)
df_pm['met_station_dist_km'] = df_pm['site_id'].map(dist_map)

In [57]:
df_final = pd.merge(
    df_pm,
    df_wind,
    left_on=['met_site_id', 'datetime'],
    right_on=['site_id', 'datetime'],
    how='left',
    suffixes=('', '_met')
)

In [58]:
# 1. Sort to ensure time order is correct for filling
df_final = df_final.sort_values(['site_id', 'datetime'])

# 2. Check missing before filling (Optional, just to see)
print("Missing before temporal fill:")
print(df_final[['wind_speed', 'wind_dir']].isna().sum())

# 3. Forward Fill (Propagate last valid observation forward)
#    We group by 'site_id' so we don't accidentally fill Site A's data into Site B
df_final['wind_speed'] = df_final.groupby('site_id')['wind_speed'].ffill()
df_final['wind_dir'] = df_final.groupby('site_id')['wind_dir'].ffill()

# 4. Backward Fill (Just in case the VERY first row is missing)
df_final['wind_speed'] = df_final.groupby('site_id')['wind_speed'].bfill()
df_final['wind_dir'] = df_final.groupby('site_id')['wind_dir'].bfill()

# 5. Final check
print("Missing after fill:")
print(df_final[['wind_speed', 'wind_dir']].isna().sum())

Missing before temporal fill:
wind_speed    44805
wind_dir      46870
dtype: int64
Missing after fill:
wind_speed    0
wind_dir      0
dtype: int64


In [59]:
df_final.head()

,county_code,site_number,poc,latitude,longitude,date_gmt,time_gmt,sample_measurement,qualifier,site_id,...,fire_id_2,fire_id_3,fire_dist_1,fire_dist_2,fire_dist_3,met_site_id,met_station_dist_km,site_id_met,wind_dir,wind_speed
0,1,13,3,37.864767,-122.302741,2020-01-01,08:00,24.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,129.3,2.8
1,1,13,3,37.864767,-122.302741,2020-01-01,09:00,27.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,142.5,4.0
2,1,13,3,37.864767,-122.302741,2020-01-01,10:00,17.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,84.1,2.0
3,1,13,3,37.864767,-122.302741,2020-01-01,11:00,14.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,190.0,2.7
4,1,13,3,37.864767,-122.302741,2020-01-01,12:00,12.0,None,06-001-0013-3,...,NaN,NaN,NaN,NaN,NaN,06-077-2010-3,92.984228,06-077-2010-3,281.0,1.9


In [60]:
df_final.to_parquet('pm25_complete_model_ready.parquet')